# Preparo

## Bibliotecas

In [12]:
import os
import sys

import pandas as pd
import plotly.express as px

In [13]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [14]:
from utils.add_country_name import add_country_name
from utils.adjust_cuisines import adjust_cuisines
from utils.clean_data import clean_data
from utils.convert_to_usd import convert_to_usd
from utils.create_color_name import create_color_name
from utils.create_price_tye import create_price_tye
from utils.create_unique_restaurant_name import create_unique_restaurant_name
from utils.rename_columns import rename_columns

## Baixando o Dataset

In [15]:
file_path = os.path.join("..", "database", "zomato.csv")

print(f"Tentando ler o arquivo em: {os.path.abspath(file_path)}")

try:
    df = pd.read_csv(file_path, 
                    encoding="utf-8", 
                    on_bad_lines="skip")
    print("Sucesso! O DataFrame foi carregado.")
except Exception as e:
    print(f"Erro: {e}")

Tentando ler o arquivo em: c:\Users\Admin\Documents\Comunidade DS\Analise de Dados Com Python\Empresa Fome Zero\database\zomato.csv
Sucesso! O DataFrame foi carregado.


## Limpando o Dataset

In [16]:
df = (df.pipe(rename_columns)
                .pipe(add_country_name)
                .pipe(clean_data)
                .pipe(convert_to_usd)
                .pipe(create_price_tye)
                .pipe(create_color_name)
                .pipe(adjust_cuisines)
                .pipe(create_unique_restaurant_name))

# Tipos Culinarios

## Top 10 Melhores Tipos de Culinárias

In [17]:
# 1. Calculando a média E a soma de votos (igual fizemos antes)
melhores_culinarias = (
    df.groupby("cuisines")
    .agg(
        media_avaliacao=("aggregate_rating", "mean"),
        quantidade_avaliacoes=("votes", "sum")
    )
    .reset_index()
)

melhores_culinarias["media_avaliacao"] = melhores_culinarias["media_avaliacao"].round(2)

# 2. Pegando o Top 10
piores_culinarias = melhores_culinarias.sort_values(
    by=["media_avaliacao", "quantidade_avaliacoes"], 
    ascending=[True, False]
)

top_10_culinarias = melhores_culinarias.head(10)

# 3. Criando o gráfico
fig = px.bar(
    top_10_culinarias, 
    x="cuisines", 
    y="media_avaliacao", 
    title="Top 10 Melhores Tipos de Culinárias",
    labels={
        "cuisines": "Tipo de Culinária", 
        "media_avaliacao": "Nota Média",
        "quantidade_avaliacoes": "Total de Votos" # Traduzindo para a caixinha do mouse ficar bonita!
    },
    text_auto=".2f", # MUDANÇA AQUI: Volta a mostrar a nota média em cima da barra
    hover_data=["quantidade_avaliacoes"], # MUDANÇA AQUI: Adiciona os votos quando passa o mouse
    color="cuisines", 
    color_discrete_sequence=px.colors.qualitative.Safe 
)

# 4. Ajustando o visual
fig.update_layout(
    xaxis_tickangle=-45, 
    title_x=0.5,
    showlegend=False, 
    xaxis_categoryorder='total descending' 
)

# 5. Exibindo o gráfico
fig.show()

## Top 10 Piores Tipos de Culinárias

In [20]:
# 1. Agrupando e calculando
piores_culinarias = (
    df.groupby("cuisines")
    .agg(
        media_avaliacao=("aggregate_rating", "mean"),
        quantidade_avaliacoes=("votes", "sum")
    )
    .reset_index()
)

# MUITO IMPORTANTE: Arredondar ANTES de ordenar. 
# Assim, notas como 2.112 e 2.114 empatam em 2.11 e acionam o desempate pelos votos.
piores_culinarias["media_avaliacao"] = piores_culinarias["media_avaliacao"].round(2)

# 2. O SEGREDO DO DESEMPATE:
# 1º: Menor nota (True) | 2º: Maior quantidade de votos (False)
piores_culinarias = piores_culinarias.sort_values(
    by=["media_avaliacao", "quantidade_avaliacoes"], 
    ascending=[True, False]
)

# Pegando as 10 piores
top_10_piores_culinarias = piores_culinarias.head(10)

# 3. Criando o gráfico
fig = px.bar(
    top_10_piores_culinarias, 
    x="cuisines", 
    y="media_avaliacao", 
    title="Top 10 Piores Tipos de Culinárias (Desempate: Mais Votos)",
    labels={
        "cuisines": "Tipo de Culinária", 
        "media_avaliacao": "Nota Média",
        "quantidade_avaliacoes": "Total de Votos" 
    },
    text_auto=".2f", 
    hover_data=["quantidade_avaliacoes"], 
    color="cuisines", 
    color_discrete_sequence=px.colors.qualitative.Safe 
)

# 4. Ajustando o visual
fig.update_layout(
    xaxis_tickangle=-45, 
    title_x=0.5,
    showlegend=False,
)

# 5. Exibindo o gráfico
fig.show()

## Top 10 Restaurantes

In [23]:
# 1. Selecionando as colunas exatas
colunas_selecionadas = [
    "restaurant_name", 
    "country", 
    "city", 
    "cuisines", 
    "average_cost_for_two_cost_in_usd", 
    "aggregate_rating", 
    "votes"
]

df_tabela = df[colunas_selecionadas]

# 2. Ordenando: 1º pela nota (maior para menor), 2º pelos votos (maior para menor)
df_ordenado = df_tabela.sort_values(
    by=["aggregate_rating", "votes"], 
    ascending=[False, False] 
)

# 3. Dicionário de tradução das colunas
traducao_colunas = {
    "restaurant_name": "Nome do Restaurante", 
    "country": "País", 
    "city": "Cidade", 
    "cuisines": "Tipo de Culinária", 
    "average_cost_for_two_cost_in_usd": "Preço para Dois (USD)", 
    "aggregate_rating": "Avaliação Média", 
    "votes": "Total de Votos"
}

# 4. Aplicando a tradução
df_traduzido = df_ordenado.rename(columns=traducao_colunas)

# 5. Exibindo a tabela no Notebook (Top 10)
df_traduzido.head(10)

,Nome do Restaurante,País,Cidade,Tipo de Culinária,Preço para Dois (USD),Avaliação Média,Total de Votos
3038,Byg Brewski Brewing Company,India,Bangalore,Continental,19.2,4.9,17394
3018,AB's - Absolute Barbecues,India,Bangalore,European,19.2,4.9,12443
3369,AB's - Absolute Barbecues,India,Chennai,BBQ,16.8,4.9,9774
4362,Barbeque Nation,India,Kolkata,North Indian,21.6,4.9,8271
2967,AB's - Absolute Barbecues,India,Bangalore,European,19.2,4.9,6877
3372,Coal Barbecues,India,Chennai,North Indian,16.8,4.9,5776
3758,Pirates of Grill,India,Gurgaon,North Indian,24.0,4.9,5760
4304,Spice Kraft,India,Kolkata,Continental,14.4,4.9,4935
6097,Sushi Hiro,Indonesia,Jakarta,Sushi,25.2,4.9,4416
1396,Café Du Monde,United States of America,New Orleans,Coffee and Tea,10.0,4.9,4036


## Melhores Restaurantes dos Principais Tipos Culinários

In [24]:
# 1. Pegando os 5 tipos culinários com MAIS restaurantes
top_5_culinarias = df["cuisines"].value_counts().head(5).index

# 2. Filtrando a base original para ter apenas essas 5 culinárias
df_top_5 = df[df["cuisines"].isin(top_5_culinarias)]

# 3. Ordenando pelas melhores notas (False) e, em caso de empate, mais votos (False)
df_top_5_ordenado = df_top_5.sort_values(
    by=["cuisines", "aggregate_rating", "votes"], 
    ascending=[True, False, False]
)

# 4. Pegando apenas o primeiro restaurante (o melhor) de cada culinária
melhores_restaurantes = df_top_5_ordenado.groupby("cuisines").first().reset_index()

# Selecionando apenas as colunas que importam para visualizar no Notebook
colunas_para_visualizar = ["cuisines", "restaurant_name", "aggregate_rating", "votes"]
melhores_restaurantes[colunas_para_visualizar]

,cuisines,restaurant_name,aggregate_rating,votes
0,American,Shake Shack,4.9,1633
1,Cafe,Tapri Central,4.9,2862
2,Italian,Darshan,4.9,3106
3,North Indian,Barbeque Nation,4.9,8271
4,Pizza,Lombardi's Pizza,4.9,1466
